In [2]:
from fastembed import TextEmbedding, LateInteractionTextEmbedding, SparseTextEmbedding 

dense_embedding_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
bm25_embedding_model = SparseTextEmbedding("Qdrant/bm25")
late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

c:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:06<00:00,  1.29s/it]


In [5]:
from langchain_docling.loader import DoclingLoader
from docling.chunking import HybridChunker
from langchain_docling.loader import ExportType

#C:\Users\crist\perso\master\2\pi\pi\data\Manuals\mds_axis_compensation_en.pdf

FILE_PATH = "./../data/Manuals/mds_axis_compensation_en.pdf"
#FILE_PATH = "https://arxiv.org/pdf/2408.09869"
MODEL = "sentence-transformers/all-MiniLM-L6-v2"

loader = DoclingLoader(
    file_path=FILE_PATH,
    export_type=ExportType.DOC_CHUNKS,
    chunker=HybridChunker(
        tokenizer=MODEL,
        max_tokens=300,
        merge_peers=True,
        repeat_table_header=True,
        omit_header_on_overflow=True,
    ),
)

docs = loader.load()

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-04-10 18:21:47,126 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-10 18:21:47,145 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-04-10 18:21:47,148 [RapidOCR] main.py:53: Using C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-04-10 18:21:47,265 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-10 18:21:47,270 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-04-10 18:21:47,270 [RapidOCR

In [7]:
documents = [doc.page_content for doc in docs]

dense_embeddings = list(dense_embedding_model.embed(documents))
bm25_embeddings = list(bm25_embedding_model.embed(documents))
late_interaction_embeddings = list(late_interaction_embedding_model.embed(documents))

In [8]:
from qdrant_client import QdrantClient

client = QdrantClient(
    url="http://localhost:6333",
    grpc_port=6334,
    prefer_grpc=True,
)

In [9]:
from qdrant_client.models import Distance, VectorParams, models

client.create_collection(
    "hybrid-search",
    vectors_config={
        "all-MiniLM-L6-v2": models.VectorParams(
            size=len(dense_embeddings[0]),
            distance=models.Distance.COSINE,
        ),
        "colbertv2.0": models.VectorParams(
            size=len(late_interaction_embeddings[0][0]),
            distance=models.Distance.COSINE,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0)  #  Disable HNSW for reranking
        ),
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF
        )
    }
)

True

In [10]:
from qdrant_client.models import PointStruct
points = []
for idx, (dense_embedding, bm25_embedding, late_interaction_embedding, doc) in enumerate(zip(dense_embeddings, bm25_embeddings, late_interaction_embeddings, documents)):
  
    point = PointStruct(
        id=idx,
        vector={
            "all-MiniLM-L6-v2": dense_embedding,
            "bm25": bm25_embedding.as_object(),
            "colbertv2.0": late_interaction_embedding,
        },
        payload={"document": doc}
    )
    points.append(point)

operation_info = client.upsert(
    collection_name="hybrid-search",
    points=points
)

In [27]:
# query = "What is the compensation for the MDS axis?"
query = "give me the structure, parameter et functionality for P-COMP-00047"

dense_vectors = next(dense_embedding_model.query_embed(query))
sparse_vectors = next(bm25_embedding_model.query_embed(query))
late_vectors = next(late_interaction_embedding_model.query_embed(query))

In [28]:
prefetch = [
        models.Prefetch(
            query=dense_vectors,
            using="all-MiniLM-L6-v2",
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(**sparse_vectors.as_object()),
            using="bm25",
            limit=20,
        ),
    ]

In [29]:
results

QueryResponse(points=[ScoredPoint(id=130, version=1, score=19.03022003173828, payload={'document': '3.3.6 Compensation of a modulo axis (P-COMP-00022)\n axis = T, R, S. Axis types, Compensation of a modulo axis = T, R, S. Dimension, Compensation of a modulo axis = T: ----. Dimension, Compensation of a modulo axis = R,S: ----. Default value, Compensation of a modulo axis = 0. Default value, Compensation of a modulo axis = 0. Remarks, Compensation of a modulo axis = . Remarks, Compensation of a modulo axis = '}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=129, version=1, score=18.158315658569336, payload={'document': '3.3.6 Compensation of a modulo axis (P-COMP-00022)\n\nDescription, Compensation of a modulo axis = This parameter defines the compensation table for a modulo axis. A modulo transition also takes place in the compensation table on the modulo transition of the axis position. The following special aspects must be noted: • The position values of the first and

In [30]:
results.points[0].payload['document']

'3.3.6 Compensation of a modulo axis (P-COMP-00022)\n axis = T, R, S. Axis types, Compensation of a modulo axis = T, R, S. Dimension, Compensation of a modulo axis = T: ----. Dimension, Compensation of a modulo axis = R,S: ----. Default value, Compensation of a modulo axis = 0. Default value, Compensation of a modulo axis = 0. Remarks, Compensation of a modulo axis = . Remarks, Compensation of a modulo axis = '

In [31]:
results = client.query_points(
         "hybrid-search",
        prefetch=prefetch,
        query=late_vectors,
        using="colbertv2.0",
        with_payload=True,
        limit=10,
)

print("Top 10 results:")
for idx in range(len(results.points)):
    print(f"Rank {idx + 1}:")
    print(f"Document: {results.points[idx].payload['document']}")
    print(f"Score: {results.points[idx].score}\n")

Top 10 results:
Rank 1:
Document: The overview of compensation parameters is sorted into a 4-column table.

 P-COMP-00005 [ } 15], Parameter = slave_ax_nr. P-COMP-00005 [ } 15], Functionality/ Short de- scription = Logical axis number of the master axis (cross compensation). P-COMP-00006 [ } 17], Structure = kw.crosscomp.table[i].. P-COMP-00006 [ } 17], Parameter = setpoint. P-COMP-00006 [ } 17], Functionality/ Short de- scription = Interpolation point of the mas- ter axis (cross compensation). P-COMP-00007 [ } 17], Structure = kw.crosscomp.table[i].. P-COMP-00007 [ } 17], Parameter = correction. P-COMP-00007 [ } 17], Functionality/ Short de- scription = Compensation values for the slave axis (cross compensation). P-COMP-00008 [ } 19], Structure = kw.crosscomp2.. P-COMP-00008 [ } 19], Parameter = unit. P-COMP-00008 [ } 19], Functionality/ Short de- scription = Unit of the length entries (plane compensation). P-COMP-00009 [ } 20], Structure =
Score: 20.693763732910156

Rank 2:
Document: